# Stage 2 batch runs on Colab (GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

One-time setup: in Google Drive, create a folder `cp_ml` and upload `CP_5G_Congestion_datasets.zip` into it.

Run the cells top to bottom. If Colab disconnects, run all cells again: finished runs are skipped, and every result is saved to Google Drive (`cp_ml/colab_out`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
# 1. Code: latest version from GitHub
%cd /content
!rm -rf repo && git clone -q https://github.com/Abhi0p2005/Congestion_Prediction.git repo
!ls repo/ml

In [ ]:
# 2. Data: unzip the dataset from Drive (once per session)
!mkdir -p /content/data && unzip -q -n /content/drive/MyDrive/cp_ml/CP_5G_Congestion_datasets.zip -d /content/data
!find /content/data -name train.csv.gz | sort

In [ ]:
# 3. Packages (PyTorch is already installed on Colab)
!pip -q install onnx onnxruntime

In [ ]:
# 4. Keep results and models on Drive, so nothing is lost if Colab disconnects.
#    Existing laptop results (from GitHub) are copied in first, so they are not re-run.
import os, shutil
OUT = '/content/drive/MyDrive/cp_ml/colab_out'
for d in ['results', 'models']:
    src, dst = f'/content/repo/ml/{d}', f'{OUT}/{d}'
    os.makedirs(dst, exist_ok=True)
    if os.path.isdir(src) and not os.path.islink(src):
        for f in os.listdir(src):
            if not os.path.exists(f'{dst}/{f}'):
                shutil.copy(f'{src}/{f}', dst)
        shutil.rmtree(src)
    if not os.path.exists(src):
        os.symlink(dst, src)
print(len(os.listdir(f'{OUT}/results')), 'result files on Drive')

In [ ]:
# 5. TCN on all experiments, 3 seeds (about 20 runs)
%cd /content/repo/ml
!python 05_run_batch.py --data /content/data --exps E1 E2 E3_k05 E3_k10 E3_k20 E4 E5 --seeds 0 1 2 --models tcn

In [ ]:
# 6. TCN with per-source scaling (E1-E4)
!python 05_run_batch.py --data /content/data --exps E1 E2 E3_k05 E3_k10 E3_k20 E4 --seeds 0 1 2 --models tcn --scaling per_source

In [ ]:
# 7. Summary table
!python 06_summarize.py

In [ ]:
# 8. Download everything new as one zip (then unzip it into C:\cp_ml\repo\ml on the laptop)
!cd /content/drive/MyDrive/cp_ml/colab_out && zip -q -r /content/colab_out.zip results models
from google.colab import files
files.download('/content/colab_out.zip')